# Word Embedding Comparison: GloVe, FastText, CLIP & ViLT

This notebook isolates the pipeline for **loading neural/behavioral data** from a picture-naming task and **comparing four families of word embeddings** in a common low-dimensional space:

| Embedding | Type | Source | Dimensionality |
|-----------|------|--------|----------------|
| **GloVe** (840B) | Static text | Pre-trained on Common Crawl | 300-D |
| **FastText** (simple) | Static text | Pre-trained on Simple Wikipedia | 300-D |
| **CLIP** (ViT-B/32) | Vision-Language | OpenAI contrastive pre-training on 400M image-text pairs | 512-D (image & text separately) |
| **ViLT** (B32-MLM) | Vision-Language | Fused image+text transformer | 768-D (CLS fused & word fused) |

## Pipeline Overview

```
1. Load raw .mat data  ──►  2. Bin & preprocess  ──►  3. Clean trials/channels
       │                                                        │
       ▼                                                        ▼
4. Lemmatize target words  ──►  5. Compute / load embeddings (GloVe, FastText, CLIP, ViLT)
                                        │
                                        ▼
                              6. PCA to 3-D  ──►  7. Interactive 3-D scatter comparison
```

### Key Distinctions Between Embeddings

- **GloVe / FastText** represent words as fixed vectors learned from text corpora. They capture distributional semantics (words that appear in similar contexts get similar vectors).
- **CLIP** learns a *shared* image-text space via contrastive learning. It produces **separate** image and text vectors that are aligned so matching pairs are close. This lets us ask: *does the picture's visual embedding land near the word's text embedding?*
- **ViLT** processes image + text *jointly* through a single transformer. It produces **fused** representations where each token (including the word) has already attended to the image. This gives us a *contextualised* word vector that incorporates visual information.

By projecting all embedding types to 3-D with PCA and colour-coding by semantic category, we can visually compare how each embedding space clusters the same word set.

## 1. Imports

In [6]:
import numpy as np
import scipy.io as sio
import os
import sys
import mat73
import collections
import pandas as pd
import pickle as pk
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from nltk.stem import WordNetLemmatizer
from torchtext.vocab import GloVe, FastText
from plotly.subplots import make_subplots
from plotly.colors import qualitative as qual

# Work from the project root so all relative paths resolve naturally.
# Guard makes this safe to re-run.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

# Local utilities
from utils.utils import reformat_raw, remove_number, get_channel_colors, plot_3d_scatter
from embeddings import MultimodalEmbedder

print('All imports OK')

All imports OK


## 2. Configuration

Edit the cell below to select the **patient** and **task**. The pipeline currently supports:
- `pictureNaming_all_data.mat` (default)
- `auditoryNaming_all_data.mat` (not used here but same loading logic)

The `shank_of_interest` list determines which electrode shanks to keep.

In [18]:
# ── Patient & task ──────────────────────────────────────────────
data_folder = 'data'
patient = 'VB'  # Change to the patient of interest
alpha = 0.05

# ── Binning ─────────────────────────────────────────────────────
bin_size = 100  # ms

# ── Channel selection ────────────────────────────────────────────
# Set to None to use all channels, or list shank prefixes to keep
shank_of_interest = None  # e.g. ['A','B','C','D','T','U','L','W','O','J']

# ── Bad channels / trials ────────────────────────────────────────
bad_channels = []  # indices to remove (0-based)
bad_trials = []    # indices to remove (0-based), or boolean mask

# ── Image folder for CLIP / ViLT embedding generation ───────────
image_folder = os.path.join(data_folder, 'pictureNaming VB')

# ── Pre-computed embedding cache (embeddings/<image-folder-name>/<model>.pk) ─
# Keyed by the image folder name so embeddings are reusable across patients
# that share the same stimulus set.
embeddings_folder = os.path.join('embeddings', os.path.basename(image_folder))
os.makedirs(embeddings_folder, exist_ok=True)

# ── Unwanted semantic categories to filter out ───────────────────
unwanted_categories = []  # e.g. ['body part']

print(f'Patient: {patient}')
print(f'Image folder:      {image_folder}')
print(f'Embeddings folder: {embeddings_folder}')

Patient: VB
Image folder:      data\pictureNaming VB
Embeddings folder: embeddings\pictureNaming VB


## 3. Load Raw Data

We load the `.mat` file produced by the data collection pipeline. The columns of `all_data` are:

| Col | Content | Type |
|-----|---------|------|
| 0 | Raw neural data (channels × time) | array |
| 1 | Trial onset (s) | float |
| 2 | Go-cue / green-screen onset (s) | float |
| 3 | Trial offset (s) | float |
| 4 | Voice onset (s) | float |
| 5 | Voice offset (s) | float |
| 6 | Target word | str |
| 7 | Answered word | str |
| 8 | Bad channels | array |
| 9 | Bad trials | array/bool |
| 10 | Sampling frequency | int |
| 11 | Channel names | list[str] |

In [19]:
# Locate the patient folder
all_patients = os.listdir(data_folder)
patient_folder_index = np.where([patient in s for s in all_patients])[0][0]
path = os.path.join(data_folder, all_patients[patient_folder_index],
                     'pictureNaming_all_data.mat')
print(f'Loading: {path}')

# Load (try mat73 first, fall back to scipy)
try:
    all_data = np.array(mat73.loadmat(path)['all_data'], dtype=object)
except Exception:
    all_data = sio.loadmat(path)['all_data']

# Remove broken / None trials
all_data = all_data[~np.array([isinstance(a, type(None)) for a in all_data[:, 0]])]
print(f'Loaded {len(all_data)} trials')

Loading: data\2023-08-16 VB\pictureNaming_all_data.mat
Loaded 203 trials


## 4. Assign Semantic Categories

Each target word belongs to a hand-labelled semantic category (e.g. *animal*, *food*, *vehicle*, etc.).  
The mapping comes from `wordset picture naming expanded.xlsx`.

In [20]:
# Load the word → category mapping
df = pd.read_excel(os.path.join(data_folder, 'wordset picture naming expanded.xlsx'))
word_column = df.columns[0]
df.set_index(word_column, inplace=True)
df_filled = df.fillna(0).apply(pd.to_numeric)
category_series = df_filled.idxmax(axis=1).reset_index()
category_series.columns = [word_column, 'Category']
word_to_category_dict = dict(zip(category_series[word_column],
                                 category_series['Category']))

print(f'Categories available: {sorted(set(word_to_category_dict.values()))}')

Categories available: ['abstract', 'animal', 'body part', 'color', 'food (exclude fruit)', 'fruit', 'homonym', 'nature', 'number', 'objects and tools', 'vehicle', 'verb']


## 5. Bin Neural Data & Extract Timing Cues

The raw data is sampled at `fs` Hz. We average within non-overlapping bins of `bin_size` ms to reduce dimensionality while preserving temporal dynamics.

```
  Raw signal (e.g. 512 Hz)   ──►  bin_size=100 ms  ──►  10 Hz effective rate
  [n_trials × n_channels × n_samples]  ──►  [n_trials × n_channels × n_bins]
```

In [21]:
fs = int(all_data[0, 10])
n_samples_per_bin = fs * bin_size // 1000

# Process target labels (remove trailing numbers from picture file names)
target_label = reformat_raw(all_data[:, 6])
target_label = np.array([remove_number(t) for t in target_label])

# Assign semantic categories & merge small groups
word_category = [word_to_category_dict[word[:].lower()] for word in target_label]
word_category = [w if w != 'fruit' and w != 'food (exclude fruit)'
                 else 'food and fruit' for w in word_category]
print(f'Category distribution: {collections.Counter(word_category)}')

# Filter unwanted categories
category_mask = np.array([wc not in unwanted_categories for wc in word_category])
all_data = all_data[category_mask]
word_category = [wc for wc in word_category if wc not in unwanted_categories]
target_label = target_label[category_mask]

# Extract timing cues
data = all_data[:, 0]
trial_onset    = reformat_raw(all_data[:, 1])
green_screen_onset = reformat_raw(all_data[:, 2])
trial_offset   = reformat_raw(all_data[:, 3])
voice_onset    = reformat_raw(all_data[:, 4])
voice_offset   = reformat_raw(all_data[:, 5])
answer_label   = reformat_raw(all_data[:, 7])

# Bin the neural data
shortest_trial = min([d.shape[0] for d in data])
data = np.array([np.array([d for d in dt[:shortest_trial]]) for dt in data]).swapaxes(1, 2)
min_length = data.shape[2] // n_samples_per_bin * n_samples_per_bin
data = data[:, :, :min_length]
data_binned = data.reshape(data.shape[0], data.shape[1], -1, n_samples_per_bin).mean(axis=3)

print(f'Sampling rate: {fs} Hz')
print(f'Binned data shape: {data_binned.shape}  (trials × channels × time_bins)')

C:\Users\Owner\AppData\Local\Temp\ipykernel_62168\2641720109.py:1: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fs = int(all_data[0, 10])


Category distribution: Counter({'abstract': 40, 'food and fruit': 37, 'animal': 36, 'body part': 32, 'nature': 31, 'objects and tools': 23, 'vehicle': 4})
Sampling rate: 2000 Hz
Binned data shape: (203, 50, 58)  (trials × channels × time_bins)


## 6. Clean Trials & Channels

Remove bad channels (e.g. noisy / broken electrodes) and bad trials (e.g. no response).  
Optionally restrict to a subset of electrode shanks.

In [22]:
# Channel names (may be absent in some datasets)
try:
    channel_names = list(all_data[0, 11])
    print(f'Channel names found: {len(channel_names)} channels')
except (IndexError, TypeError):
    channel_names = None
    print('No channel names available — shank filtering will be skipped')

# Remove bad channels
clean_data_binned = np.delete(data_binned, bad_channels, axis=1)
if channel_names is not None:
    clean_channel_names = [cn for i, cn in enumerate(channel_names) if i not in bad_channels]
    print(f'Removed {len(bad_channels)} bad channel(s), {len(clean_channel_names)} remaining')
else:
    clean_channel_names = None

# Handle bad trials (can be a boolean mask or index list)
if len(bad_trials) > 0 and isinstance(bad_trials[0], (bool, np.bool_)):
    # Boolean mask: True = keep
    clean_data_binned   = clean_data_binned[bad_trials]
    clean_target_label  = np.squeeze(target_label[bad_trials])
    clean_answer_label  = answer_label[bad_trials]
    clean_word_category = np.array(word_category)[bad_trials]
    print(f'Removed {np.sum(~np.array(bad_trials))} bad trial(s) via boolean mask')
else:
    clean_data_binned   = np.delete(clean_data_binned, bad_trials, axis=0)
    clean_target_label  = np.delete(target_label, bad_trials)
    clean_answer_label  = np.delete(answer_label, bad_trials)
    clean_word_category = np.delete(np.array(word_category), bad_trials)
    print(f'Removed {len(bad_trials)} bad trial(s) by index')

# Optional: filter by shank (only possible when channel names are available)
if shank_of_interest is not None:
    if clean_channel_names is not None:
        shank_mask = [i for i, cn in enumerate(clean_channel_names)
                      if cn[0] in shank_of_interest]
        clean_data_binned   = clean_data_binned[:, shank_mask, :]
        clean_channel_names = [clean_channel_names[i] for i in shank_mask]
        print(f'Shank filter {shank_of_interest}: kept {len(shank_mask)} channel(s)')
    else:
        print('Warning: shank_of_interest is set but channel names are unavailable — skipping shank filter')

n_clean_trials   = clean_data_binned.shape[0]
n_clean_channels = clean_data_binned.shape[1]
print(f'Clean data: {n_clean_trials} trials × {n_clean_channels} channels')

No channel names available — shank filtering will be skipped
Removed 0 bad trial(s) by index
Clean data: 203 trials × 50 channels


## 7. Lemmatize Target Words

We lemmatize every target word to its **noun base form** (e.g. *dogs → dog*, *leaves → leaf*).  
This ensures consistent look-up across embedding vocabularies and avoids duplicates due to inflection.

In [23]:
lemmatizer = WordNetLemmatizer()
target_lexeme = np.array([
    lemmatizer.lemmatize(''.join([c for c in w if c.isalpha()]), pos='n')
    for w in clean_target_label
])

print(f'Unique lemmatised words: {len(np.unique(target_lexeme))}')
print(f'Examples: {target_lexeme[:10]}')

Unique lemmatised words: 53
Examples: ['nut' 'pear' 'moose' 'bear' 'bean' 'lime' 'newt' 'cat' 'fish' 'soup']


## 8. Compute / Load Word Embeddings

We obtain four embedding families for every target word:

### Static text embeddings (word-level look-up)
- **GloVe 840B** (300-D): trained on 840 billion tokens of web text.
- **FastText Simple** (300-D): trained on Simple English Wikipedia; can handle unseen words via subword information.

### Multimodal (vision-language) embeddings
- **CLIP ViT-B/32**: returns *separate* 512-D vectors for an image and its caption. The two spaces are aligned through a contrastive objective (cosine similarity is maximised for matching image–text pairs). We store both `image` and `text` embeddings.
- **ViLT B32-MLM**: processes image and text tokens through a *single* transformer, producing:
  - `cls_fused` — the [CLS] token's output, summarising the joint image+text representation.
  - `word_fused` — the average of the subword tokens of the target word, after attending to the image.

The multimodal embeddings are pre-computed and stored as pickle files.  
If the pickle files are missing, the cell will generate them from the image folder using `embeddings.MultimodalEmbedder`.

In [24]:
# ── 8a. Static text embeddings ──────────────────────────────────
embed_ft = FastText(language='simple')
target_word_embed_FastText = [embed_ft[w].numpy() for w in target_lexeme]

embed_glove = GloVe(dim=300, name='840B')
target_word_embed_GloVe = [embed_glove[w].numpy() for w in target_lexeme]

print(f'GloVe  : {np.array(target_word_embed_GloVe).shape}')
print(f'FastText: {np.array(target_word_embed_FastText).shape}')

GloVe  : (203, 300)
FastText: (203, 300)


In [25]:
# ── 8b. CLIP embeddings (image + text) ────────────────────────────
clip_pkl = os.path.join(embeddings_folder, 'clip-vit-b32.pk')

if os.path.exists(clip_pkl):
    print(f'Loading pre-computed CLIP embeddings from {clip_pkl}')
    with open(clip_pkl, 'rb') as f:
        clip_results = pk.load(f)
else:
    print('Generating CLIP embeddings from images (this may take a moment)...')
    embedder = MultimodalEmbedder(backend='clip')
    clip_results = embedder.embed_folder(image_folder)
    with open(clip_pkl, 'wb') as f:
        pk.dump(clip_results, f)
    print(f'Saved to {clip_pkl}')

# Map each target word to its CLIP vectors
clip_words_arr = np.array(clip_results['words'])
target_word_embed_CLIP_images = []
target_word_embed_CLIP_words  = []
for t in target_lexeme:
    idx = np.where(clip_words_arr == t.lower())[0]
    target_word_embed_CLIP_images.append(clip_results['image'][idx].squeeze())
    target_word_embed_CLIP_words.append(clip_results['text'][idx].squeeze())

print(f'CLIP image: {np.array(target_word_embed_CLIP_images).shape}')
print(f'CLIP text : {np.array(target_word_embed_CLIP_words).shape}')

Loading pre-computed CLIP embeddings from embeddings\pictureNaming VB\clip-vit-b32.pk
CLIP image: (203, 512)
CLIP text : (203, 512)


In [26]:
# ── 8c. ViLT embeddings (cls_fused + word_fused) ──────────────────────
vilt_pkl = os.path.join(embeddings_folder, 'vilt-b32-mlm.pk')

if os.path.exists(vilt_pkl):
    print(f'Loading pre-computed ViLT embeddings from {vilt_pkl}')
    with open(vilt_pkl, 'rb') as f:
        vilt_results = pk.load(f)
else:
    print('Generating ViLT embeddings from images (this may take a moment)...')
    embedder = MultimodalEmbedder(backend='vilt')
    vilt_results = embedder.embed_folder(image_folder)
    with open(vilt_pkl, 'wb') as f:
        pk.dump(vilt_results, f)
    print(f'Saved to {vilt_pkl}')

# Map each target word to its ViLT vectors
vilt_words_arr = np.array(vilt_results['words'])
target_word_embed_VILT_images = []
target_word_embed_VILT_words  = []
for t in target_lexeme:
    idx = np.where(vilt_words_arr == t.lower())[0]
    target_word_embed_VILT_images.append(vilt_results['cls_fused'][idx].squeeze())
    target_word_embed_VILT_words.append(vilt_results['word_fused'][idx].squeeze())

print(f'ViLT cls_fused : {np.array(target_word_embed_VILT_images).shape}')
print(f'ViLT word_fused: {np.array(target_word_embed_VILT_words).shape}')

Loading pre-computed ViLT embeddings from embeddings\pictureNaming VB\vilt-b32-mlm.pk
ViLT cls_fused : (203, 768)
ViLT word_fused: (203, 768)


## 9. Reduce to 3-D with PCA & Visualise

We use PCA to project each embedding type down to 3 dimensions for interactive visualisation.  
Points are **colour-coded by semantic category** so you can compare how tightly each embedding space clusters semantically related words.

### What to look for
- **GloVe / FastText**: words from the same category (*animals*, *tools*, etc.) should form loose clusters if the text-only distributional statistics reflect semantics.
- **CLIP image**: clusters may reflect *visual* similarity rather than purely semantic grouping (e.g. *can* might cluster with cylindrical objects regardless of category).
- **CLIP text**: should look similar to static embeddings but through the lens of a contrastive loss.
- **ViLT fused**: these embeddings have *already* mixed visual and textual information, so clusters might look different from either modality alone.

In [ ]:
pca = PCA(3)

target_low_D_GloVe       = pca.fit_transform(np.array(target_word_embed_GloVe))
target_low_D_FastText    = pca.fit_transform(np.array(target_word_embed_FastText))
target_low_D_CLIP_img    = pca.fit_transform(np.array(target_word_embed_CLIP_images))
target_low_D_CLIP_txt    = pca.fit_transform(np.array(target_word_embed_CLIP_words))
target_low_D_VILT_cls    = pca.fit_transform(np.array(target_word_embed_VILT_images))
target_low_D_VILT_word   = pca.fit_transform(np.array(target_word_embed_VILT_words))

print('PCA projections computed for all 6 embedding variants.')

In [ ]:
# ── GloVe ────────────────────────────────────────────────────────
fig1 = plot_3d_scatter(target_low_D_GloVe, clean_word_category,
                       labels=target_lexeme,
                       title='GloVe (840B, 300-D) — text only')
fig1.show()

In [ ]:
# ── FastText ─────────────────────────────────────────────────────
fig2 = plot_3d_scatter(target_low_D_FastText, clean_word_category,
                       labels=target_lexeme,
                       title='FastText (Simple Wikipedia, 300-D) — text only')
fig2.show()

In [ ]:
# ── CLIP image embeddings ────────────────────────────────────────
fig3 = plot_3d_scatter(target_low_D_CLIP_img, clean_word_category,
                       labels=target_lexeme,
                       title='CLIP ViT-B/32 — image embeddings (512-D)')
fig3.show()

In [ ]:
# ── CLIP text embeddings ─────────────────────────────────────────
fig4 = plot_3d_scatter(target_low_D_CLIP_txt, clean_word_category,
                       labels=target_lexeme,
                       title='CLIP ViT-B/32 — text embeddings (512-D)')
fig4.show()

In [ ]:
# ── ViLT CLS-fused (image-level) ────────────────────────────────
fig5 = plot_3d_scatter(target_low_D_VILT_cls, clean_word_category,
                       labels=target_lexeme,
                       title='ViLT B32 — CLS fused (768-D, image+text joint)')
fig5.show()

In [ ]:
# ── ViLT word-fused (word-level) ────────────────────────────────
fig6 = plot_3d_scatter(target_low_D_VILT_word, clean_word_category,
                       labels=target_lexeme,
                       title='ViLT B32 — Word fused (768-D, word attended to image)')
fig6.show()

## 10. Quantitative Comparison: Inter-Embedding Similarity

Beyond visual inspection, we compute pairwise **Representational Similarity Analysis (RSA)** to quantify how similarly each embedding space organises words.

For each embedding type we build an $N \times N$ **Representational Dissimilarity Matrix (RDM)** using cosine distance, then correlate (Spearman) every pair of RDMs.  
High correlation ⟹ the two embedding spaces impose a similar geometry on the same word set.

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
import plotly.graph_objects as go

# Collect all full-dimensional embeddings into a dict
all_embeds = {
    'GloVe':        np.array(target_word_embed_GloVe),
    'FastText':     np.array(target_word_embed_FastText),
    'CLIP image':   np.array(target_word_embed_CLIP_images),
    'CLIP text':    np.array(target_word_embed_CLIP_words),
    'ViLT cls':     np.array(target_word_embed_VILT_images),
    'ViLT word':    np.array(target_word_embed_VILT_words),
}

# Build RDMs (upper-triangle vectors) using cosine distance
rdms = {}
for name, emb in all_embeds.items():
    rdms[name] = pdist(emb, metric='cosine')

# Compute pairwise Spearman correlations between RDMs
embed_names = list(rdms.keys())
n_embed = len(embed_names)
rsa_matrix = np.ones((n_embed, n_embed))

for i in range(n_embed):
    for j in range(i + 1, n_embed):
        rho, _ = spearmanr(rdms[embed_names[i]], rdms[embed_names[j]])
        rsa_matrix[i, j] = rho
        rsa_matrix[j, i] = rho

# Plot as heatmap
fig_rsa = go.Figure(data=go.Heatmap(
    z=rsa_matrix,
    x=embed_names,
    y=embed_names,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(rsa_matrix, 3),
    texttemplate='%{text}',
    textfont={'size': 12},
))
fig_rsa.update_layout(
    title='RSA: Pairwise Spearman Correlation of Embedding RDMs',
    width=700, height=600,
    xaxis_title='Embedding', yaxis_title='Embedding',
)
fig_rsa.show()

print('\nPairwise Spearman correlations between embedding RDMs:')
print(pd.DataFrame(rsa_matrix, index=embed_names, columns=embed_names).round(3))

## Interpretation Guide

### RSA Heatmap
- **GloVe ↔ FastText**: typically high positive correlation — both are distributional text models trained on similar corpora.
- **CLIP image ↔ CLIP text**: moderate–high correlation — CLIP explicitly aligns these two spaces.
- **ViLT cls ↔ ViLT word**: moderate correlation — both are fused but at different granularity (global vs. token-level).
- **Text-only ↔ Multimodal**: the *gap* here is informative: if CLIP/ViLT embeddings correlate weakly with GloVe, it suggests the visual modality introduces structure not present in text alone.

### 3-D Scatter Plots
- **Tight clusters by colour** ⟹ the embedding space respects semantic categories.
- **Overlapping clusters** ⟹ the embedding conflates categories (or the categories are genuinely similar in that representation).
- **Outliers** (single words far from their cluster) may signal polysemy or poor embedding coverage.

### Next Steps
These embeddings can serve as **regression targets** for neural decoding:
- See `semantics.ipynb` for KernelRidge regression from neural activity → embeddings.
- See `cross_semantic_regression.ipynb` for cross-task generalisation of the regression.